# 03 - Seleção de Features

No notebook 01, a matriz de correlação já mostrou alguns pares de indicadores bem
redundantes. Aqui decido, com um pouco mais de cuidado, quais features realmente valem a pena
manter para a clusterização.

Isso importa porque o K-Means usa distância euclidiana: se duas colunas carregam basicamente
a mesma informação, o efeito prático é dar peso dobrado para aquele conceito no cálculo da
distância.

## Importando bibliotecas

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
 
from src.preprocessing import TODAS_AS_FEATURES, dataframe_padronizado
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

## Visualizando as correlações

In [2]:
df = pd.read_csv('../data/raw/dataset_paises.csv')
corr = df[TODAS_AS_FEATURES].corr()

pares_fortes = []
for i, a in enumerate(TODAS_AS_FEATURES):
    for j, b in enumerate(TODAS_AS_FEATURES):
        if i < j and abs(corr.loc[a, b]) > 0.75:
            pares_fortes.append((a, b, round(corr.loc[a, b], 2)))

pd.DataFrame(pares_fortes, columns=['variavel_1', 'variavel_2', 'correlacao'])

,variavel_1,variavel_2,correlacao
0,mortalidade_infantil,expectativa_vida,-0.89
1,mortalidade_infantil,fertilidade_total,0.85
2,renda,pib_per_capita,0.90
3,expectativa_vida,fertilidade_total,-0.76


Três pares se destacam:

- `mortalidade_infantil` × `expectativa_vida` (-0,89)
- `mortalidade_infantil` × `fertilidade_total` (0,85)
- `renda` × `pib_per_capita` (0,90)

Para cada par, a pergunta é: essas duas colunas representam **conceitos diferentes** (mesmo
que correlacionados na prática) ou são **a mesma coisa medida de dois jeitos**?

- `mortalidade_infantil`, `expectativa_vida` e `fertilidade_total` são conceitualmente distintos: uma é sobrevivência infantil, outra é longevidade da população em geral, outra é comportamento reprodutivo. Andam juntas porque refletem o mesmo nível de desenvolvimento, mas cada uma ainda possui um significado próprio;

- `renda` (renda líquida por pessoa) e `pib_per_capita` (PIB total dividido pela população) são duas formas de medir quase a mesma coisa: quanto dinheiro corresponde a cada pessoa naquele país.

## Decisão

Baseando-se nas análises foi removido **`renda`** e mantido **`pib_per_capita`**, que é a medida mais padronizada internacionalmente para comparar o tamanho da economia por pessoa entre países. O modelo de clusterização passa a usar **8 features** em vez de 9.

Antes de bater o martelo, vale testar: será que essa mudança realmente altera o resultado, ou
só deixa o modelo mais enxuto sem perder informação?

In [3]:
features_selecionadas = [f for f in TODAS_AS_FEATURES if f != 'renda']

Xs_todas = StandardScaler().fit_transform(df[TODAS_AS_FEATURES])
Xs_selecionadas = StandardScaler().fit_transform(df[features_selecionadas])

labels_todas = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(Xs_todas)
labels_selecionadas = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(Xs_selecionadas)

print(f"Silhouette com 9 features: {silhouette_score(Xs_todas, labels_todas):.3f}")
print(f"Silhouette com 8 features (sem renda): {silhouette_score(Xs_selecionadas, labels_selecionadas):.3f}")
print(f"Concordância entre as duas versões (ARI): {adjusted_rand_score(labels_todas, labels_selecionadas):.3f}")

Silhouette com 9 features: 0.283
Silhouette com 8 features (sem renda): 0.285
Concordância entre as duas versões (ARI): 0.912


O silhouette score com 8 features fica igual e a concordância com a versão de 9 features é de 0,91, os grupos formados são quase idênticos;

Ou seja, tirar `renda` deixa o modelo mais simples e sem a redundância, sem perder capacidade de separar os países.

## Salvando a seleção

In [4]:
with open('../data/processed/features_selecionadas.json', 'w') as f:
    json.dump(features_selecionadas, f, ensure_ascii=False, indent=2)

df_padronizado_sel, scaler_sel = dataframe_padronizado(df, features_selecionadas)
df_padronizado_sel.to_csv('../data/processed/paises_padronizados_selecionado.csv', index=False)
corr.to_csv('../data/processed/matriz_correlacao.csv')

print("Features selecionadas:", features_selecionadas)

Features selecionadas: ['mortalidade_infantil', 'exportacoes', 'saude', 'importacoes', 'inflacao', 'expectativa_vida', 'fertilidade_total', 'pib_per_capita']
